# 🌌 Deepfake Detection AI — Free Google Colab T4 GPU Hosting

Run the complete **Multi-Modal Deepfake Forensic Platform** on a **Free NVIDIA Tesla T4 GPU (15 GB VRAM)** with a public HTTPS link accessible worldwide.

### ⚡ Quick Setup Instructions (Takes 1-2 minutes):
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** under Hardware accelerator, then click **Save**.
3. Run the cells below in order from **Step 1 to Step 5**.

### 1. Verify NVIDIA GPU & CUDA Acceleration
Confirms NVIDIA driver, CUDA availability, and allocated GPU VRAM.

In [ ]:
!nvidia-smi

import torch
print("\n" + "=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
    print(f"Device Count:    {torch.cuda.device_count()}")
    print(f"VRAM Allocated:  {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
else:
    print("⚠️ GPU not detected! Please go to Runtime -> Change runtime type -> T4 GPU")
print("=" * 50)

### 2. Clone Project Repository & Verify Checkpoints
Clones the codebase into `/content/deepfake_app`, ensures directory structures, and verifies that the fine-tuned ~109 MB model weights (`backend/checkpoints/best_model.pth`) are fully present (with automatic download fallback if cloned as a Git LFS pointer).

In [ ]:
import os
import shutil

APP_DIR = "/content/deepfake_app"

# Clean previous run if needed
if os.path.exists(APP_DIR):
    print(f"[*] Cleaning existing {APP_DIR} for fresh deployment...")
    shutil.rmtree(APP_DIR)

# Clone the repository (supports GitHub or Hugging Face Spaces)
print("[*] Cloning repository...")
!git clone https://github.com/Chamansengar/Deepfake_detection_Web.git {APP_DIR} || \
 git clone https://huggingface.co/spaces/Chaman-sengar07/deepfake-detection-ai {APP_DIR}

%cd {APP_DIR}

# Ensure required backend directories exist
os.makedirs(f"{APP_DIR}/backend/runs", exist_ok=True)
os.makedirs(f"{APP_DIR}/backend/checkpoints", exist_ok=True)

# Verify model checkpoint file size (at least 50 MB to detect Git LFS pointer stubs)
ckpt_path = f"{APP_DIR}/backend/checkpoints/best_model.pth"
min_valid_size = 50 * 1024 * 1024

needs_download = False
if not os.path.exists(ckpt_path):
    print("⚠️ Checkpoint file missing!")
    needs_download = True
elif os.path.getsize(ckpt_path) < min_valid_size:
    print(f"⚠️ Checkpoint file size ({os.path.getsize(ckpt_path)} bytes) is an LFS pointer stub!")
    needs_download = True

if needs_download:
    print("[*] Downloading full fine-tuned model checkpoint (~109 MB) from Hugging Face...")
    !wget -q --show-progress -O {ckpt_path} \
        https://huggingface.co/spaces/Chaman-sengar07/deepfake-detection-ai/resolve/main/backend/checkpoints/best_model.pth

if os.path.exists(ckpt_path):
    file_size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
    print(f"✅ Model checkpoint ready: {ckpt_path} ({file_size_mb:.2f} MB)")
else:
    print("⚠️ Checkpoint could not be retrieved. App will use base VGGFace2 weights.")

!ls -lh {APP_DIR}/backend/checkpoints

### 3. Install System & Python Dependencies (GPU-Optimized)
Installs multimedia codecs (`ffmpeg`, `libsndfile1`) and all required Python packages for FastAPI, MTCNN face detection, Grad-CAM, and audio transformers without modifying Colab's pre-installed CUDA PyTorch.

In [ ]:
# 1. Install system codecs silently
print("[*] Installing system audio/video codecs (ffmpeg, libsndfile)...")
!apt-get install -y -qq --no-install-recommends ffmpeg libsndfile1 > /dev/null 2>&1

# 2. Install facenet-pytorch & ttach with --no-deps (uses Colab's pre-installed PyTorch & Pillow)
print("[*] Installing facenet-pytorch & ttach...")
!pip install -q --no-deps facenet-pytorch ttach

# 3. Install backend dependencies from pre-compiled wheels (bypasses all Pillow compilation)
print("[*] Installing Python packages (FastAPI, Transformers, Librosa, etc.)...")
!pip install -q --only-binary :all: \
    fastapi \
    uvicorn \
    python-multipart \
    opencv-python-headless \
    "transformers>=4.38.0" \
    "huggingface-hub>=0.23.0,<1.0" \
    librosa \
    soundfile \
    httpx

# 4. Zero-build install for pytorch-grad-cam
!rm -rf /tmp/gradcam
!git clone -q https://github.com/jacobgil/pytorch-grad-cam.git /tmp/gradcam && cp -r /tmp/gradcam/pytorch_grad_cam /content/deepfake_app/backend/ && cp -r /tmp/gradcam/pytorch_grad_cam /content/deepfake_app/

# 5. Verify backend imports & GPU detection
print("\n[*] Verifying backend module loading...")
!python -c "import sys; sys.path.insert(0, '/content/deepfake_app'); from backend import app; print('✅ Backend successfully imported! Device:', app.device, '| Audio Detector:', app.audio_detector_available)"

### 4. Setup Cloudflare Tunnel (Free Worldwide HTTPS Tunnel)
Downloads the official `cloudflared` binary into `/usr/local/bin/cloudflared`. Cloudflare Tunnel gives you a free, public HTTPS URL without needing an account, auth token, or port forwarding.

In [ ]:
print("[*] Downloading Cloudflare Tunnel binary...")
!wget -q -nc -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version
print("✅ Cloudflare Tunnel binary ready.")

### 5. Launch FastAPI Backend & Expose Public URL 🚀
Starts the FastAPI app with GPU acceleration on port 8000 and exposes it globally through Cloudflare Tunnel.

**Keep this cell running** to keep your application online. To stop the server, click the Colab Stop button.

In [ ]:
import os
import sys
import time
import re
import subprocess

APP_DIR = "/content/deepfake_app"
os.chdir(APP_DIR)

# Terminate any previously running server or tunnel
!fuser -k 8000/tcp 2>/dev/null || true
!killall cloudflared 2>/dev/null || true
time.sleep(1)

# Ensure runs directory exists
os.makedirs(f"{APP_DIR}/backend/runs", exist_ok=True)

# 1. Start FastAPI application in background
print("[*] Starting Deepfake Detection Engine (FastAPI + GPU)...")
env = os.environ.copy()
env["PYTHONPATH"] = f"{APP_DIR}:{APP_DIR}/backend:" + env.get("PYTHONPATH", "")

server_cmd = [
    sys.executable, "-m", "uvicorn", "backend.app:app",
    "--host", "0.0.0.0",
    "--port", "8000",
    "--workers", "1"
]
server_process = subprocess.Popen(
    server_cmd,
    cwd=APP_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 2. Start Cloudflare Tunnel in background
print("[*] Launching Cloudflare Tunnel...")
tunnel_log = "/tmp/cloudflared.log"
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)

tunnel_cmd = [
    "/usr/local/bin/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8000",
    "--logfile", tunnel_log,
    "--metrics", "127.0.0.1:45678"
]
tunnel_process = subprocess.Popen(
    tunnel_cmd,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 3. Discover public Cloudflare HTTPS URL
public_url = None
print("[*] Waiting for public tunnel URL...")
for _ in range(45):
    time.sleep(1)
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r", errors="ignore") as f:
            log_data = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_data)
            if match:
                public_url = match.group(0)
                break

# Detect Colab internal kernel proxy port (optional backup URL)
colab_url = None
try:
    from google.colab.output import eval_js
    colab_url = eval_js("google.colab.kernel.proxyPort(8000)")
except Exception:
    pass

print("\n" + "=" * 70)
if public_url:
    print("🎉 DEEPFAKE DETECTION AI IS LIVE IN PRODUCTION!")
    print(f"👉 Public Web App:     {public_url}")
    print(f"👉 Interactive Docs:   {public_url}/docs")
    print(f"👉 API Status Endpoint: {public_url}/api/status")
else:
    print("⚠️ Could not automatically parse Cloudflare URL. Check logs below.")

if colab_url:
    print(f"👉 Colab Internal URL:  {colab_url}")
print("=" * 70 + "\n")
print("ℹ️ Streaming server inference logs below (Stop cell to shut down):\n")

# 4. Stream live logs to console
try:
    while True:
        line = server_process.stdout.readline()
        if line:
            print(line, end="")
        elif server_process.poll() is not None:
            print("[!] Server process terminated unexpectedly.")
            break
        time.sleep(0.01)
except KeyboardInterrupt:
    print("\n[*] Shutting down server and tunnel gracefully...")
    server_process.terminate()
    tunnel_process.terminate()
    print("✅ Server stopped.")

### 6. Verify System Health & API Status (Optional Test)
Queries the `/api/status` endpoint to confirm GPU device assignment, checkpoint metrics (AUC/accuracy), and model availability.

In [ ]:
import httpx
import json

try:
    resp = httpx.get("http://127.0.0.1:8000/api/status", timeout=10.0)
    print(f"Status Code: {resp.status_code}")
    print("Backend Response:")
    print(json.dumps(resp.json(), indent=2))
except Exception as e:
    print(f"Error querying backend: {e}")